In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from typing import List
from tqdm import tqdm
import random
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
from sklearn.preprocessing import LabelEncoder
import scipy.sparse as sp

c:\Users\kanze\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_users = pd.read_csv('data_original/data_original/users.csv')
df_interactions = pd.read_csv('data_original/data_original/interactions.csv')
df_items = pd.read_csv('data_original/data_original/items.csv')

In [3]:
df_interactions.drop(df_interactions.tail(2).index, inplace=True)

In [4]:
# Конвектируем время в datetime64
df_interactions = pd.read_csv('data_original/data_original/interactions.csv')
df_interactions['last_watch_dt'] = pd.to_datetime(df_interactions['last_watch_dt'], errors='coerce')

In [5]:
# Добавляем новый столбец completed (0<60 , 1>60 просмотрено )
df_interactions['completed'] = (df_interactions['watched_pct'] >= 60).astype(int)


In [6]:
# df_interactions.dropna(subset=['user_id','item_id', 'last_watch_dt', 'watched_pct', 'total_dur','completed'])
# df_users.dropna(subset=['user_id','age', 'income', 'sex', 'kids_flg'])
# df_items.dropna(subset=['item_id', 'content_type', 'title', 'title_orig', 'release_year', 'genres', 'countries', 'for_kids', 'age_rating', 'studios', 'directors', 'actors', 'description', 'keywords'])

In [7]:
print(df_interactions.isna().sum())

user_id            0
item_id            0
last_watch_dt      0
total_dur          0
watched_pct      828
completed          0
dtype: int64


In [8]:
df_interactions = df_interactions[np.isfinite(df_interactions["watched_pct"])]

In [9]:
test_size_days=10

In [10]:
from datetime import datetime, timedelta

# Тестовый промежуток времени равен 10 дней
max_date = df_interactions['last_watch_dt'].max()
test_start = max_date - timedelta(days=test_size_days)

In [11]:
# Разделение на тренировочный и тестовый
df_interactions_train = df_interactions[df_interactions['last_watch_dt'] < test_start]
df_interactions_test = df_interactions[df_interactions['last_watch_dt'] >= test_start]

In [12]:
df_interactions = df_interactions.dropna(subset=['user_id','item_id', 'last_watch_dt', 'watched_pct', 'total_dur','completed'])
# df_users=df_users.dropna(subset=['user_id','age', 'income', 'sex', 'kids_flg'])
# df_items=df_items.dropna(subset=['item_id', 'content_type', 'title', 'title_orig', 'release_year', 'genres', 'countries', 'for_kids', 'age_rating', 'studios', 'directors', 'actors', 'description', 'keywords'])

In [13]:
# df_interactions = df_interactions.dropna(subset=["user_id", "item_id", "watched_pct", "completed"], how="any")

In [14]:
print(df_interactions.isna().sum())
print(df_users.isna().sum())
print(df_items.isna().sum())

user_id          0
item_id          0
last_watch_dt    0
total_dur        0
watched_pct      0
completed        0
dtype: int64
user_id         0
age         14095
income      14776
sex         13831
kids_flg        0
dtype: int64
item_id             0
content_type        0
title               0
title_orig       4745
release_year       98
genres              0
countries          37
for_kids        15397
age_rating          2
studios         14898
directors        1509
actors           2619
description         2
keywords          423
dtype: int64


In [15]:
df_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15963 entries, 0 to 15962
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   item_id       15963 non-null  int64  
 1   content_type  15963 non-null  object 
 2   title         15963 non-null  object 
 3   title_orig    11218 non-null  object 
 4   release_year  15865 non-null  float64
 5   genres        15963 non-null  object 
 6   countries     15926 non-null  object 
 7   for_kids      566 non-null    float64
 8   age_rating    15961 non-null  float64
 9   studios       1065 non-null   object 
 10  directors     14454 non-null  object 
 11  actors        13344 non-null  object 
 12  description   15961 non-null  object 
 13  keywords      15540 non-null  object 
dtypes: float64(3), int64(1), object(10)
memory usage: 1.7+ MB


In [16]:
df_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 840197 entries, 0 to 840196
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   user_id   840197 non-null  int64 
 1   age       826102 non-null  object
 2   income    825421 non-null  object
 3   sex       826366 non-null  object
 4   kids_flg  840197 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 32.1+ MB


In [17]:
df_interactions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5475423 entries, 0 to 5476250
Data columns (total 6 columns):
 #   Column         Dtype         
---  ------         -----         
 0   user_id        int64         
 1   item_id        int64         
 2   last_watch_dt  datetime64[ns]
 3   total_dur      int64         
 4   watched_pct    float64       
 5   completed      int32         
dtypes: datetime64[ns](1), float64(1), int32(1), int64(3)
memory usage: 271.5 MB


In [18]:
last_interactions = df_interactions.tail(10)

In [19]:
last_interactions

,user_id,item_id,last_watch_dt,total_dur,watched_pct,completed
5476241,1073802,9927,2021-08-07,6425,97.0,1
5476242,268216,3071,2021-04-21,5752,98.0,1
5476243,497899,9629,2021-05-29,45,1.0,0
5476244,438585,7829,2021-08-02,6804,100.0,1
5476245,786732,4880,2021-05-12,753,0.0,0
5476246,648596,12225,2021-08-13,76,0.0,0
5476247,546862,9673,2021-04-13,2308,49.0,0
5476248,697262,15297,2021-08-20,18307,63.0,1
5476249,384202,16197,2021-04-19,6203,100.0,1
5476250,319709,4436,2021-08-15,3921,45.0,0


In [20]:
user_ids = last_interactions['user_id']
item_ids = last_interactions['item_id']
complete = last_interactions['completed']

In [21]:
from tqdm import tqdm
import time
import numpy as np

def track_progress_tqdm(iterable, desc=None):
    """
    Обертка для отслеживания прогресса с помощью tqdm
    
    Parameters:
    iterable: итерируемый объект
    desc: описание прогресс-бара
    
    Returns:
    Обернутый итератор с прогресс-баром
    """
    return tqdm(iterable, desc=desc, ncols=100, ascii=True)

# Пример использования
def process_data(data):
    results = []
    for item in track_progress_tqdm(data, desc="Обработка данных"):
        # Имитация работы
        time.sleep(0.1)
        results.append(item * 2)
    return results

# Использование
data = list(range(50))
processed = process_data(data)

Обработка данных: 100%|#############################################| 50/50 [00:05<00:00,  9.90it/s]


In [29]:
class ALS:
    def __init__(self, n_factors=10, alpha=40, regularization=0.1, iterations=15):
        """
        Инициализация ALS модели
        
        Parameters:
        - n_factors: количество латентных факторов
        - alpha: параметр уверенности для неявной обратной связи
        - regularization: коэффициент регуляризации (lambda)
        - iterations: количество итераций обучения
        """
        self.n_factors = n_factors
        self.alpha = alpha
        self.regularization = regularization
        self.iterations = iterations
        self.user_factors = None
        self.item_factors = None
        self.loss_history = []
    
    def fit(self, ratings):
        """
        Обучение ALS модели
        
        Parameters:
        - ratings: scipy sparse matrix в формате (users × items)
        """
        n_users, n_items = ratings.shape
        
        # Инициализация матриц факторов
        self.user_factors = np.random.normal(0, 0.1, (n_users, self.n_factors))
        self.item_factors = np.random.normal(0, 0.1, (n_items, self.n_factors))
        
        print("Начинаем обучение ALS...")
        
        for iteration in range(self.iterations):
            # Шаг 1: Фиксируем item factors, обновляем user factors
            for u in range(n_users):
                # Получаем индексы и значения оценок пользователя
                user_ratings = ratings[u].toarray().flatten()
                rated_items = np.where(user_ratings > 0)[0]
                
                if len(rated_items) > 0:
                    # Вычисляем уверенности для неявной обратной связи
                    confidence = 1 + self.alpha * user_ratings[rated_items]
                    
                    # Матрица item factors для оцененных items
                    Y = self.item_factors[rated_items]
                    
                    # Вычисляем матрицу A и вектор b
                    weighted_Y_T = Y.T * confidence  # (n_factors, n_rated_items)
                    A = weighted_Y_T @ Y + self.regularization * np.eye(self.n_factors) # (n_factors, n_factors)
                    b = Y.T @ confidence # (n_factors,)
                    
                    # Решаем линейную систему
                    self.user_factors[u] = np.linalg.solve(A, b)
            
            # Шаг 2: Фиксируем user factors, обновляем item factors
            for i in range(n_items):
                # Получаем индексы и значения оценок для item
                item_ratings = ratings[:, i].toarray().flatten()
                rating_users = np.where(item_ratings > 0)[0]
                
                if len(rating_users) > 0:
                    # Вычисляем уверенности
                    confidence = 1 + self.alpha * item_ratings[rating_users]
                    
                    # Матрица user factors для оценивших пользователей
                    X = self.user_factors[rating_users]
                    
                    # Вычисляем матрицу A и вектор b
                    weighted_X_T = X.T * confidence  # (n_factors, n_rated_users)
                    A = weighted_X_T @ X + self.regularization * np.eye(self.n_factors) # (n_factors, n_factors)
                    b = X.T @ confidence # (n_factors,)
                    
                    # Решаем линейную систему
                    self.item_factors[i] = np.linalg.solve(A, b)
            
            # Вычисляем loss
            loss = self._compute_loss(ratings)
            self.loss_history.append(loss)
            print(f"Итерация {iteration + 1}/{self.iterations}, Loss: {loss:.4f}")
    
    def _compute_loss(self, ratings):
        """Вычисление функции потерь"""
        loss = 0
        n_users, n_items = ratings.shape
        
        for u in range(n_users):
            for i in range(n_items):
                if ratings[u, i] > 0:
                    prediction = self.user_factors[u] @ self.item_factors[i]
                    confidence = 1 + self.alpha * ratings[u, i]
                    error = confidence - prediction
                    loss += error ** 2
        
        # Добавляем регуляризацию
        reg_loss = self.regularization * (
            np.sum(self.user_factors ** 2) + np.sum(self.item_factors ** 2)
        )
        
        return loss + reg_loss
    
    def predict(self, user_id, item_id):
        """Предсказание оценки"""
        return self.user_factors[user_id] @ self.item_factors[item_id]
    
    def recommend(self, user_id, n_recommendations=10):
        """Рекомендации для пользователя"""
        user_vector = self.user_factors[user_id]
        scores = user_vector @ self.item_factors.T
        top_items = np.argsort(scores)[::-1][:n_recommendations]
        return top_items, scores[top_items]
    
    def plot_loss(self):
        """Визуализация истории потерь"""
        plt.figure(figsize=(10, 6))
        plt.plot(self.loss_history, marker='o')
        plt.title('История обучения ALS')
        plt.xlabel('Итерация')
        plt.ylabel('Loss')
        plt.grid(True)
        plt.show()

In [30]:
def precision(recommended_list, bought_list):
    recommended = np.array(recommended_list)
    bought = np.array(bought_list)

    # флаги: какие рекомендованные товары действительно куплены
    flags = np.isin(recommended, bought)

    return flags.sum() / len(recommended)


def precision_at_k(recommended_list, bought_list, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)

    flags = np.isin(recommended, bought)

    return flags.sum() / k


def ap_k(recommended_list, bought_list, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)

    # релевантность: рекомендованный товар куплен или нет
    flags = np.isin(recommended, bought)

    # если нет ни одного релевантного — AP = 0
    if flags.sum() == 0:
        return 0.0

    sum_precision = 0.0

    for i in range(k):
        if flags[i]:
            # precision@i+1 (на префиксе)
            precision_i = np.isin(recommended[:i+1], bought).sum() / (i + 1)
            sum_precision += precision_i

    # нормируем на число релевантных объектов в топ-k
    return sum_precision / flags.sum()


def map_k(recommended_list, bought_list, k=5, u=1):
    
    if u == 1:
        return ap_k(recommended_list[u-1], bought_list[u-1], k=5)
    
    sum = 0
    for i in range(0, u):
        ap_k_map = ap_k(recommended_list[i], bought_list[i], k=5)
        sum += ap_k_map

    result = sum / u
    
    return result

In [31]:
def create_matrix(df, user_col='user_id', item_col='item_id', 
                           value_col='watched_pct'):

    unique_users = sorted(set(df[user_col]))
    unique_items = sorted(set(df[item_col]))

    user_to_index = {user: idx for idx, user in enumerate(unique_users)}
    item_to_index = {item: idx for idx, item in enumerate(unique_items)}

    index_to_user = {idx: user for user, idx in user_to_index.items()}
    index_to_item = {idx: item for item, idx in item_to_index.items()}

    n_users = len(unique_users)
    n_items = len(unique_items)

    user_indices = [user_to_index[u] for u in df[user_col]]
    item_indices = [item_to_index[i] for i in df[item_col]]

    sparse_matrix = csr_matrix((df[value_col], (user_indices, item_indices)), shape=(n_users, n_items))
    
    matrix_df = pd.DataFrame.sparse.from_spmatrix(
        sparse_matrix,
        index=unique_users,
        columns=unique_items
    )
    
    return matrix_df, sparse_matrix, index_to_user, index_to_item, user_to_index, item_to_index 

train_df, train_matrix, index_to_user_train, index_to_item_train, user_to_index_train, item_to_index_train = create_matrix(
    df_interactions_train, user_col='user_id', item_col='item_id', value_col='watched_pct'
)
test_df, test_matrix, index_to_user_test, index_to_item_test, user_to_index_test, item_to_index_test  = create_matrix(
    df_interactions_test, user_col='user_id', item_col='item_id', value_col='watched_pct'
)



In [32]:
print("\n" + "="*50)
als_model = ALS(
        n_factors=25,
        alpha=20,
        regularization=0.01,
        iterations=5 # Уменьшил для тестирования
    )
    
# Обучение модели
try:
    als_model.fit(train_matrix)
    
    # Визуализация процесса обучения
    als_model.plot_loss()
except MemoryError as e:
    print(f"Ошибка памяти даже после оптимизации: {e}")
    print("Возможно, данные все еще слишком велики для вашей системы.")
    print("Рассмотрите фильтрацию пользователей/итемов или снижение размерности факторов.")


Начинаем обучение ALS...


KeyboardInterrupt: 

In [ ]:

# print("\n" + "="*50)
# als_model = ALS(
#         n_factors=25,
#         alpha=20,
#         regularization=0.01,
#         iterations=15
#     )
    
# # Обучение модели
# als_model.fit(train_matrix)
    
# # Визуализация процесса обучения
# als_model.plot_loss()
    
# # Оценка модели
# print("\nОценка модели:")
    
# # Предсказания на тестовой выборке
# test_predictions = []
# test_actual = []
    
# for u in range(test_matrix.shape[0]):
#     for i in range(test_matrix.shape[1]):
#         if test_matrix[u, i] > 0:
#                 pred = als_model.predict(u, i)
#                 test_predictions.append(pred)
#                 test_actual.append(test_matrix[u, i])
    
# # Вычисляем метрики
# mse = mean_squared_error(test_actual, test_predictions)
# rmse = np.sqrt(mse)


# print(f"RMSE на тестовой выборке: {rmse:.4f}")




Начинаем обучение ALS...


MemoryError: Unable to allocate 73.0 GiB for an array with shape (98961, 98961) and data type float64

In [ ]:
data_test_group = data_test.groupby(data_test['user_id'])['item_id'].unique().reset_index()
data_test_group.columns = ['user_id', 'item_id']
data_test_group

In [ ]:
userids = data_test_group['user_id'].values
 
userids_test = np.arange(len(userids))

In [ ]:


def recommendations__original_ids(user_matrix_index, item_recommendations):
    original_item_ids = [index_to_item_train[idx] for idx in item_recommendations]
    return user_matrix_index, original_item_ids



In [ ]:
recommended = []
for i in range(len(userids_test)):
    recommended_items, scores = als_model.recommend(userids_test[i], n_recommendations=10)
    recommended.append({'item_id_pred': recommended_items})

result = pd.DataFrame(recommended, userids).reset_index()
result

In [ ]:
recom =[]
for i in range(len(result)):
    user, recommend = recommendations__original_ids(userids[i], result['item_id_pred'][i])
    recom.append({'item_id_pred': recommend})

result_pred = pd.DataFrame(recom, userids).reset_index()
result_pred

In [ ]:


print('als_map_k =', map_k(result_pred['item_id_pred'], data_test_group['item_id'], k=10, u=len(data_test_group)))

